# Fraud screening: does 99% accuracy mean useful detection?

This public sample reports aggregate accuracy on a rare-event target.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path('../public/fraud_rare_event.csv')
df = pd.read_csv(data_path)
positive_rate = df['fraud'].mean()
print(f'Rows: {len(df)}')
print(f'Positive rate: {positive_rate:.4%}')

Rows: 6000
Positive rate: 1.0833%


## Reported evaluation

The notebook uses a stratified holdout but reports only accuracy.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['fraud', 'case_id'])
y = df['fraud']
categorical = ['channel']
numeric = [column for column in X.columns if column not in categorical]
preprocess = ColumnTransformer([
    ('numeric', StandardScaler(), numeric),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2603, stratify=y
)
model = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(class_weight={0: 1, 1: 3}, random_state=2603)),
])
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, prediction):.6f}')

Test accuracy: 0.988667
Majority baseline accuracy: 0.989333
Majority baseline recall: 0.000000


**Learner claim to test:** Nearly 99% test accuracy proves this classifier catches rare fraud.